# Sprint 4 — Uplift Models: S-, T-, and X-Learners
**Customer Targeting & Incremental Revenue Optimization**

We model each customer's **individual treatment effect** (CATE) — how much the email
changes *that person's* behaviour — using three meta-learners, then let Qini (Sprint 5)
pick the winner.

**Outcome choice — an honest, important decision.**
We first targeted **conversion** (the business-critical approx 1% purchase event). But the
uplift *signal* there proved too sparse for any model to reliably beat random (verified
by Qini in Sprint 5). The bottleneck was **outcome density**, not algorithm strength:
a rare event leaves the per-customer treatment effect buried in noise. So we model the
denser **visit** outcome (~15% base rate) — where the heterogeneity signal is
detectable — and keep conversion as the ultimate business goal that visiting leads to.
*Visit is the modeling proxy; conversion is the goal.*

- **S-learner**: one model, treatment as a feature; predict twice, difference.
- **T-learner**: separate treated & control models; difference predictions.
- **X-learner** (hand-rolled): imputes counterfactual effects crisscross + propensity
  blend — robust to the 2:1 treatment:control imbalance.

> Run Sprints 2–3 first (`data/processed/hillstrom_clean.csv`).

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC = ROOT / "data" / "processed"
sys.path.append(str(ROOT / "src"))
from uplift_models import SLearner, TLearner, XLearner

df = pd.read_csv(PROC / "hillstrom_clean.csv")
print("Loaded:", df.shape)

Loaded: (64000, 13)


In [2]:
# Base learner — swap here to test algorithms.
# Default: GradientBoosting. To try XGBoost, install xgboost and uncomment.
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor

def make_clf(): return GradientBoostingClassifier(random_state=0)
def make_reg(): return GradientBoostingRegressor(random_state=0)

# --- XGBoost option (pip install xgboost) ---
# from xgboost import XGBClassifier, XGBRegressor
# def make_clf(): return XGBClassifier(random_state=0, eval_metric="logloss", n_estimators=200)
# def make_reg(): return XGBRegressor(random_state=0, n_estimators=200)
print("Base learner:", make_clf().__class__.__name__)

Base learner: GradientBoostingClassifier


## 1. Features & primary outcome

Outcome = **conversion** (the business-meaningful, stable binary metric).
We use the raw `history` (drop `history_segment` — same info, avoids redundancy),
and one-hot encode the categorical covariates.

In [3]:
OUTCOME = "visit"   # modeling proxy (dense). Conversion = ultimate goal (sparse).
y = df[OUTCOME].astype(int).values
w = df["treatment"].astype(int).values

feature_cols = ["recency", "history", "mens", "womens", "newbie",
                "zip_code", "channel"]
X = pd.get_dummies(df[feature_cols], drop_first=True)
print("Feature matrix:", X.shape)
print("Features:", X.columns.tolist())

Feature matrix: (64000, 9)
Features: ['recency', 'history', 'mens', 'womens', 'newbie', 'zip_code_Surburban', 'zip_code_Urban', 'channel_Phone', 'channel_Web']


## 2. Train/test split — stratified on treatment

Stratifying on `w` keeps the treatment:control ratio identical in train and test.

In [4]:
X_tr, X_te, w_tr, w_te, y_tr, y_te = train_test_split(
    X, w, y, test_size=0.30, random_state=42, stratify=w
)
print(f"Train: {len(X_tr):,}  | treatment share {w_tr.mean():.3f}")
print(f"Test:  {len(X_te):,}  | treatment share {w_te.mean():.3f}")
print("Ratios match across splits => stratification worked.")

Train: 44,800  | treatment share 0.667
Test:  19,200  | treatment share 0.667
Ratios match across splits => stratification worked.


## 3. Build the three learners

In [5]:
# S-learner
s = SLearner(base=make_clf()).fit(X_tr, w_tr, y_tr)
u_s = s.predict_uplift(X_te)
print("S-learner done.  mean uplift = %.5f" % u_s.mean())

S-learner done.  mean uplift = 0.06118


In [6]:
# T-learner
t = TLearner(base_factory=make_clf).fit(X_tr, w_tr, y_tr)
u_t = t.predict_uplift(X_te)
print("T-learner done.  mean uplift = %.5f" % u_t.mean())

T-learner done.  mean uplift = 0.06386


### X-learner — the algorithm (implemented from scratch)

1. **Outcome models:** fit `mu_t` on treated, `mu_c` on control (like the T-learner).
2. **Impute effects (crisscross):** for each *treated* customer, effect ≈ `Y − mu_c(X)`;
   for each *control* customer, effect ≈ `mu_t(X) − Y`. Now every customer has an
   *imputed* treatment-effect label — the quantity that is never directly observed.
3. **Effect models:** fit regressors `tau_t`, `tau_c` on those imputed effects.
4. **Blend:** `tau(x) = g·tau_c(x) + (1−g)·tau_t(x)`, where `g` = P(treated).
   The propensity weight makes it lean on the better-estimated side — which is why
   it is robust to the treatment:control imbalance.

In [7]:
# X-learner — hand-rolled from the meta-learner equations (src/uplift_models.py).
# Expected to handle the 2:1 imbalance best via the propensity-weighted blend.
x = XLearner(outcome_factory=make_clf, effect_factory=make_reg).fit(X_tr, w_tr, y_tr)
u_x = x.predict_uplift(X_te)
print("X-learner done.  mean uplift = %.5f" % u_x.mean())

X-learner done.  mean uplift = 0.06385


## 4. Sanity check: do the uplift scores behave?

A quick gut-check before formal Qini evaluation (Sprint 5): the mean predicted
uplift should be roughly in the neighborhood of the ATE we measured (~0.005), and
the score distributions should spread out (a model that gives everyone the same
score has found no heterogeneity to exploit).

In [8]:
scores = pd.DataFrame({"S": u_s, "T": u_t, "X": u_x})
print("Mean predicted uplift by model (compare to visit ATE ~0.06):")
print(scores.mean())
print("\nSpread (std) of predicted uplift — bigger = more heterogeneity captured:")
print(scores.std())
print("\n% of customers predicted to be HARMED (negative uplift = sleeping dogs):")
print((scores < 0).mean().mul(100).round(1))

Mean predicted uplift by model (compare to visit ATE ~0.06):
S    0.061179
T    0.063861
X    0.063853
dtype: float64

Spread (std) of predicted uplift — bigger = more heterogeneity captured:
S    0.030815
T    0.041822
X    0.032121
dtype: float64

% of customers predicted to be HARMED (negative uplift = sleeping dogs):
S    0.3
T    1.6
X    0.8
dtype: float64


**Result — all three models are well-behaved.** Mean predicted uplift (S/T/X ≈
0.061/0.064/0.064) sits right at the visit ATE (~0.06) — all three are correctly
calibrated. Spreads differ in character: the T-learner is widest (0.042, boldest/
noisiest), the X-learner tightest (0.032, the propensity blend regularizing toward
stability). Predicted "sleeping dogs" are few (0.3–1.6%) and the models broadly agree.
These checks are necessary but **not** sufficient — they say nothing about which model
*ranks* customers best. That verdict is Qini (Sprint 5).

In [9]:
# stash test-set scores + labels for Sprints 5 & 6
# y here is the MODELING outcome (visit). We also save the true conversion for
# Sprint 6's revenue calc, aligned to the same test rows.
out = X_te.copy()
out["treatment"] = w_te
out["visit"] = y_te                                   # modeling outcome (what models predict)
out["conversion"] = df.loc[X_te.index, "conversion"].values   # business goal (for $ value)
out["uplift_S"] = u_s
out["uplift_T"] = u_t
out["uplift_X"] = u_x
out.to_csv(PROC / "uplift_scores_test.csv", index=False)
print("Saved test-set uplift scores (+visit +conversion) -> data/processed/uplift_scores_test.csv")
print("Sprint 4 complete -> Sprint 5: Qini evaluation picks the winner.")

Saved test-set uplift scores (+visit +conversion) -> data/processed/uplift_scores_test.csv
Sprint 4 complete -> Sprint 5: Qini evaluation picks the winner.
